In [5]:
import pandas as pd
import numpy as np

# Cargar dataset limpio de licitaciones
df = pd.read_csv("Licitaciones_Andalucia_Perfecto.csv", sep=";")

# Convertir importe a numérico
df["Importe"] = (
    df["Importe"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df["Importe"] = pd.to_numeric(df["Importe"], errors="coerce")

# Convertir fechas
df["Fecha_Adjudicacion"] = pd.to_datetime(df["Fecha_Adjudicacion"], errors="coerce")
df["Fecha_Publicacion"] = pd.to_datetime(df["Fecha_Publicacion"], errors="coerce")

# Limpiar empresas no identificables
empresas_invalidas = [
    "VER RESOLUCIÓN ADJUNTA",
    "VER RESOLUCION ADJUNTA",
    "ENTIDADES RELACIONADAS EN EL ANEXO II DE LA RESOLUCIÓN DE ADJUDICACIÓN",
    "ENTIDADES RELACIONADAS EN EL ANEXO II DE LA RESOLUCION DE ADJUDICACION"
]

df["Empresa"] = df["Empresa"].astype(str).str.strip().str.upper()

df_limpio = df[
    (~df["Empresa"].isin(empresas_invalidas)) &
    (df["Empresa"].notna()) &
    (df["Importe"].notna()) &
    (df["Importe"] > 0)
].copy()

# Crear variables temporales
df_limpio["Anio"] = df_limpio["Fecha_Adjudicacion"].dt.year
df_limpio["Mes"] = df_limpio["Fecha_Adjudicacion"].dt.to_period("M").astype(str)
df_limpio["Trimestre"] = df_limpio["Fecha_Adjudicacion"].dt.to_period("Q").astype(str)

# Exportar dataset limpio
df_limpio.to_csv("licitaciones_limpias.csv", index=False, encoding="utf-8-sig")

# DATASET CREACION DE RANKING DE EMPRESAS

ranking_empresas = (
    df_limpio
    .groupby(["Empresa", "NIF"], dropna=False)
    .agg(
        Importe_Total=("Importe", "sum"),
        Numero_Contratos=("Titulo", "count"),
        Importe_Medio=("Importe", "mean")
    )
    .reset_index()
    .sort_values("Importe_Total", ascending=False)
)

ranking_empresas.to_csv("ranking_empresas.csv", index=False, encoding="utf-8-sig")

# EVOLUCIÓN TEMPROAL

licitaciones_temporal = (
    df_limpio
    .groupby("Mes")
    .agg(
        Importe_Total=("Importe", "sum"),
        Numero_Contratos=("Titulo", "count"),
        Importe_Medio=("Importe", "mean")
    )
    .reset_index()
    .sort_values("Mes")
)

licitaciones_temporal.to_csv("licitaciones_temporal.csv", index=False, encoding="utf-8-sig")

# Añadir clasificación manual de cotizadas

mapa_tickers = {
    "ENDESA ENERGIA SA": ("ELE.MC", "BME", "Sí"),
    "PFIZER S.L": ("PFE", "NYSE", "Sí - matriz"),
    "ABBOTT LABORATORIES S.A": ("ABT", "NYSE", "Sí - matriz"),
    "BERKSHIRE HATHAWAY EUROPEAN INSURANCE DESIGNATED ACTIVITY COMPANY SUCURSAL EN ESPAÑA": ("BRK-B", "NYSE", "Sí - matriz"),
    "MEDTRONIC IBERICA SA": ("MDT", "NYSE", "Sí - matriz"),
    "TELEFONICA DE ESPAÑA SAU": ("TEF.MC", "BME", "Sí"),
    "CONSTRUCCIONES Y AUXILIAR DE FERROCARRILES S.A": ("CAF.MC", "BME", "Sí")
}

ranking_empresas["Ticker"] = ranking_empresas["Empresa"].map(
    lambda x: mapa_tickers.get(x, ("", "", "No"))[0]
)

ranking_empresas["Mercado"] = ranking_empresas["Empresa"].map(
    lambda x: mapa_tickers.get(x, ("", "", "No"))[1]
)

ranking_empresas["Cotizada"] = ranking_empresas["Empresa"].map(
    lambda x: mapa_tickers.get(x, ("", "", "No"))[2]
)

ranking_empresas.to_csv("ranking_empresas_cotizadas.csv", index=False, encoding="utf-8-sig")



